In [ ]:
import matplotlib.pyplot as plt
from typing import Literal
import numpy as np
from utils import *

KAPLAN_COLOR = INNOVATIONS['kaplan']['color']
CHINCHILLA_COLOR = INNOVATIONS['chinchilla']['color']


E, A, B = 1.69, 406.4, 410.7
alpha, beta = .34, .28
G = (alpha * A/(beta * B)) ** (1/(alpha + beta))


a_K, b_K = .73, .27
a_C, b_C =  .46, .54


In [ ]:
def chinchilla_loss(N, D, irr_loss = False):
    return (0 if not irr_loss else E) + A/(N ** alpha) + B/(D ** beta)

kaplan_n_constant = 3.6331917579105387e-06 #9e-7

def N_opt(C, paper = Literal['kaplan', 'chinchila']):
    if (paper == 'kaplan'):
        return kaplan_n_constant * C**a_K
    elif paper == 'chinchilla':
        return G * (C/6) ** a_C

def D_opt(C, paper = Literal['kaplan', 'chinchila']):
    if (paper == 'kaplan'):
        return 1/(6 * kaplan_n_constant) * C**b_K
    elif paper == 'chinchilla':
        return (1/G) * (C/6) ** b_C


def chinchilla_optimal(C):
    return chinchilla_loss(N_opt(C, 'chinchilla'), D_opt(C, 'chinchilla')) 



def kaplan_optimal(C):
    return chinchilla_loss(N_opt(C, 'kaplan'), D_opt(C, 'kaplan')) 

In [ ]:
Cs = [10 ** i for i in range(15, 30)]

kaplan_losses = list(map(kaplan_optimal, Cs))
chinchilla_losses = list(map(chinchilla_optimal, Cs))

plt.figure(figsize=(10, 5))

line1 = plt.plot(Cs, chinchilla_losses, linewidth=2.5, color=CHINCHILLA_COLOR)
line2 = plt.plot(Cs, kaplan_losses, linewidth=2.5, color=KAPLAN_COLOR)

print(chinchilla_losses)
print(kaplan_losses)
print(np.array(kaplan_losses) / np.array(chinchilla_losses))

chinchilla_final = chinchilla_losses[-1]
kaplan_final = kaplan_losses[-1]
max_c = Cs[-1]

plt.plot([max_c * 0.995, max_c], [chinchilla_final, chinchilla_final], 
         color=CHINCHILLA_COLOR, linewidth=1.5, alpha=0.8, clip_on=False)
plt.plot([max_c, max_c * 1.5], [chinchilla_final, chinchilla_final], 
         color=CHINCHILLA_COLOR, linewidth=1, alpha=0.7, clip_on=False)
plt.text(max_c * 1.6, chinchilla_final, 'Chinchilla Scaling', 
         color=CHINCHILLA_COLOR, fontsize=12, va='center', fontweight='bold', clip_on=False)

plt.plot([max_c * 0.995, max_c], [kaplan_final, kaplan_final], 
         color=KAPLAN_COLOR, linewidth=1.5, alpha=0.8, clip_on=False)
plt.plot([max_c, max_c * 1.5], [kaplan_final, kaplan_final], 
         color=KAPLAN_COLOR, linewidth=1, alpha=0.7, clip_on=False)
plt.text(max_c * 1.6, kaplan_final, 'Kaplan Scaling', 
         color=KAPLAN_COLOR, fontsize=12, va='center', fontweight='bold', clip_on=False)

plt.ylabel("Validation Loss - Irreducible", fontsize=14)
plt.xlabel("Compute (FLOPs)", fontsize=14)
plt.title("Switching from Kaplan to Chinchilla Scaling\nhas Scale Dependent Improvements", fontsize=14, fontweight='bold')

plt.xlim(float(Cs[0]), float(Cs[-1]))
plt.ylim(1e-2, 1e2)
plt.yscale('log')
plt.xscale('log')
plt.grid(True, which='major', alpha=0.5, linestyle='-', linewidth=0.5)
plt.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('figures/kaplan_to_chinchilla.png', dpi=600, bbox_inches='tight')

In [ ]:
import math
import numpy as np

def find_x_for_y(xs, ys, target_y, log_x=True, log_y=True):
    """
    Given ordered lists xs and ys, find the x value that corresponds to target_y
    using linear interpolation/extrapolation in linear or log space.
    
    Args:
        xs: list of x values (should be ordered)
        ys: list of y values corresponding to xs
        target_y: the y value to find the corresponding x for
        log_x: if True, interpolate in log(x) space
        log_y: if True, interpolate in log(y) space
    
    Returns:
        x value corresponding to target_y
    """
    xs = list(xs)
    ys = list(ys)
    
    # Handle edge cases
    if len(xs) != len(ys):
        raise ValueError("xs and ys must have the same length")
    if len(xs) < 2:
        raise ValueError("Need at least 2 points for interpolation")
    
    # Check for positive values if using log scaling
    if log_x and any(x <= 0 for x in xs):
        raise ValueError("All x values must be positive when using log_x=True")
    if log_y and (any(y <= 0 for y in ys) or target_y <= 0):
        raise ValueError("All y values and target_y must be positive when using log_y=True")
    
    # Transform to log space if needed
    if log_x:
        xs_work = [math.log(x) for x in xs]
    else:
        xs_work = xs[:]
    
    if log_y:
        ys_work = [math.log(y) for y in ys]
        target_y_work = math.log(target_y)
    else:
        ys_work = ys[:]
        target_y_work = target_y
    
    # Find the appropriate segment for interpolation/extrapolation
    if target_y_work <= min(ys_work):
        # Use first two points for extrapolation
        x1, x2 = xs_work[0], xs_work[1]
        y1, y2 = ys_work[0], ys_work[1]
    elif target_y_work >= max(ys_work):
        # Use last two points for extrapolation
        x1, x2 = xs_work[-2], xs_work[-1]
        y1, y2 = ys_work[-2], ys_work[-1]
    else:
        # Find the two points that bracket target_y_work
        for i in range(len(ys_work) - 1):
            if ys_work[i] <= target_y_work <= ys_work[i + 1] or ys_work[i] >= target_y_work >= ys_work[i + 1]:
                x1, x2 = xs_work[i], xs_work[i + 1]
                y1, y2 = ys_work[i], ys_work[i + 1]
                break
    
    # Linear interpolation/extrapolation in the (possibly transformed) space
    if y2 == y1:  # Avoid division by zero
        x_target_work = x1
    else:
        x_target_work = x1 + (target_y_work - y1) * (x2 - x1) / (y2 - y1)
    
    # Transform back from log space if needed
    if log_x:
        x_target = math.exp(x_target_work)
    else:
        x_target = x_target_work
    
    return x_target

In [ ]:
compute_ratios, valid_computes = [], []

for k_loss, C in zip(kaplan_losses, Cs):
    ratio = C / find_x_for_y(Cs, chinchilla_losses, k_loss)
    compute_ratios.append(ratio)
    valid_computes.append(C)


# Plot the result
plt.figure(figsize=(10, 6))
plt.plot(valid_computes, compute_ratios, 'o-', color=CHINCHILLA_COLOR, label='C_Kaplan / C_Chinchilla')
plt.xscale('log')
plt.ylabel("Compute Ratio")
plt.xlabel("Kaplan Compute (FLOPs)")
plt.title("Compute Ratio: Kaplan vs Chinchilla for Same Loss")
plt.grid(True, alpha=0.5)
plt.yscale('log')
plt.show()